# Phase 3 — Fine-tune on IndicDLP (Colab)

Picks up from the Phase 2 pretrained 9-class checkpoint and runs:
1. **Pseudo-labeling** of BaDLAD-unlabeled with the pretrained model
2. **CBST** class-balanced thresholds (rare classes not swamped)
3. **Self-training rounds** on {real + pseudo}
4. **Fine-tune on IndicDLP** (re-headed to IndicDLP's ontology) — *added next*

Logic lives in `src/finetuning/self_training.py`; these cells only orchestrate.
**Runtime:** T4 is fine for the smoke test (Cell 4). Switch to **A100** for the full run (Cell 5), off-peak.

## Cell 0 — Bootstrap (mount Drive, install deps)

In [1]:
# Runtime -> Change runtime type -> A100 (full run) or T4 (smoke test) -> Save FIRST
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


subprocess.run(['pip','install','-q','ultralytics','huggingface_hub','pyyaml',
                'pycocotools','kagglehub','tqdm'], check=True)
subprocess.run(['pip','install','-q',
                'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT_ROOT:', PROJECT_ROOT)

Mounted at /content/drive
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
PROJECT_ROOT: /content/drive/MyDrive/doclayout-yolo-indic


## Cell 1 — Sync code from GitHub (plain files, no zip)
Clones the repo, auto-detects `src/` (works whether it's at the repo root or inside a
wrapper dir like `doclayout-yolo-indic/`), and copies it to `PROJECT_ROOT/src` on Drive so
`config.py`'s `__file__`-anchored paths resolve to Drive (persistent outputs/checkpoints).
**Prereq:** commit `src/` as plain files (drop the zip) and add
`src/finetuning/self_training.py` + `__init__.py`.

In [2]:
import subprocess, shutil
from pathlib import Path

GITHUB_URL    = 'https://github.com/vigneshpalanivelr/mtech-project-aiml.git'
GITHUB_BRANCH = 'main'
REPO_LOCAL    = Path('/content/_repo')

shutil.rmtree(REPO_LOCAL, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','-b',GITHUB_BRANCH,
                GITHUB_URL, str(REPO_LOCAL)], check=True)

# Find src/ wherever it lives: repo root, or one level down (wrapper dir).
candidates = [REPO_LOCAL/'src'] + sorted(REPO_LOCAL.glob('*/src'))
SRC = next((c for c in candidates if (c/'config.py').exists()), None)
assert SRC, f'Could not find src/config.py under {REPO_LOCAL}'
BASE = SRC.parent
print('Found project at:', BASE)

# Copy code (not docs) to PROJECT_ROOT so REPO_ROOT = parents[1] -> Drive.
for item in ['src','tests','requirements.txt','README.md']:
    s = BASE/item; d = PROJECT_ROOT/item
    if s.is_dir():   shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    elif s.exists(): shutil.copy(s, d)

assert (PROJECT_ROOT/'src'/'finetuning'/'self_training.py').exists(), \
    'Commit src/finetuning/self_training.py + __init__.py to the repo first!'
print('Code synced (plain files) ->', PROJECT_ROOT/'src')

Found project at: /content/_repo/doclayout-yolo-indic
Code synced (plain files) -> /content/drive/MyDrive/doclayout-yolo-indic/src


## Cell 6 - IndicDLP class check

In [3]:
%cd {PROJECT_ROOT}
from src.finetuning.train_finetuning import stage_indicdlp, read_classes

# need_train=0 -> download val fully, skip train entirely (just the class check)
raw = stage_indicdlp('VigneshPR/IndicDLP', '/content/IndicDLP_raw', need_train=0)
names, id_to_idx = read_classes(raw/'annotations'/'instances_val2017.json')
print(f"\n{len(names)} classes:")
print(names)

/content/drive/MyDrive/doclayout-yolo-indic


annotations/instances_train2017.json:   0%|          | 0.00/378M [00:00<?, ?B/s]

annotations/instances_val2017.json:   0%|          | 0.00/47.1M [00:00<?, ?B/s]

05:12:45 | INFO    | doclayout_indic.train_finetuning | Downloading val2017.tar ...


val2017.tar:   0%|          | 0.00/5.39G [00:00<?, ?B/s]

05:14:08 | INFO    | doclayout_indic.train_finetuning | Extracting val2017 ...
05:14:23 | INFO    | doclayout_indic.train_finetuning | val2017: 23268 images
05:14:23 | INFO    | doclayout_indic.train_finetuning | need_train=0 -> skipping train2017 download
05:14:24 | INFO    | doclayout_indic.train_finetuning | IndicDLP has 42 classes: advertisement, answer, author, chapter-title, contact-info, dateline, figure, figure-caption ...

42 classes:
['advertisement', 'answer', 'author', 'chapter-title', 'contact-info', 'dateline', 'figure', 'figure-caption', 'first-level-question', 'flag', 'folio', 'footer', 'footnote', 'formula', 'header', 'headline', 'index', 'jumpline', 'options', 'ordered-list', 'page-number', 'paragraph', 'placeholder-text', 'quote', 'reference', 'second-level-question', 'section-title', 'sidebar', 'sub-headline', 'sub-ordered-list', 'sub-section-title', 'subsub-ordered-list', 'subsub-section-title', 'sub-unordered-list', 'subsub-headline', 'subsub-unordered-list', 'tab

## Cell 7 - Fine-tune on IndicDLP

In [4]:
%cd {PROJECT_ROOT}
from src.finetuning.train_finetuning import run_finetuning

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

SELF_TRAINED = PROJECT_ROOT/'output'/'self_training_real'/'round_2'/'train'/'weights'/'best.pt'
assert SELF_TRAINED.exists(), f"self-trained checkpoint not found: {SELF_TRAINED}"

best = run_finetuning(
    self_trained = SELF_TRAINED,
    hf_repo      = 'VigneshPR/IndicDLP',
    drive_out    = PROJECT_ROOT/'data'/'indicdlp_yolo',
    local_root   = '/content/indicdlp_yolo',
    work_dir     = PROJECT_ROOT/'output'/'finetune_indicdlp',
    train_cap    = 20000,   # subset of 95K for compute
)
print('Final Phase 3 model:', best)

/content/drive/MyDrive/doclayout-yolo-indic
05:14:30 | INFO    | doclayout_indic.train_finetuning | val2017: 23268 images


train2017.tar.part-aa:   0%|          | 0.00/5.37G [00:00<?, ?B/s]

05:16:40 | INFO    | doclayout_indic.train_finetuning | train2017: 1 parts -> 24162 complete images (need 22000)
05:16:49 | INFO    | doclayout_indic.train_finetuning | IndicDLP has 42 classes: advertisement, answer, author, chapter-title, contact-info, dateline, figure, figure-caption ...
05:16:51 | INFO    | doclayout_indic.train_finetuning | val2017: 23268 source images present on disk
05:17:08 | INFO    | doclayout_indic.train_finetuning |   val2017: materialised 2000/3000
05:17:14 | INFO    | doclayout_indic.train_finetuning | val2017: 3000 labels (46584 boxes), 3000 images, 0 failed
05:17:27 | INFO    | doclayout_indic.train_finetuning | train2017: 24162 source images present on disk
05:17:50 | INFO    | doclayout_indic.train_finetuning |   train2017: materialised 2000/12082
05:17:59 | INFO    | doclayout_indic.train_finetuning |   train2017: materialised 4000/12082
05:18:18 | INFO    | doclayout_indic.train_finetuning |   train2017: materialised 6000/12082
05:18:29 | INFO    | d

/usr/local/lib/python3.12/dist-packages/doclayout_yolo/utils/checks.py:641: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


AMP: checks passed ✅


...

3000
                   all       3000      46584      0.552       0.47      0.472      0.327

      Epoch    GPU_mem     box_om     cls_om     dfl_om     box_oo     cls_oo     dfl_oo  Instances       Size


      30/30      41.1G     0.8135     0.8298      1.065     0.8997      1.007      1.086        299       1024: 100%|██████████| 755/755 [06:05<00:00,  2.06it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:35<00:00,  2.64it/s]


3000
                   all       3000      46584       0.54      0.472      0.474      0.328

30 epochs completed in 3.528 hours.
Optimizer stripped from /content/drive/MyDrive/doclayout-yolo-indic/output/finetune_indicdlp/finetune/weights/last.pt, 40.7MB
Optimizer stripped from /content/drive/MyDrive/doclayout-yolo-indic/output/finetune_indicdlp/finetune/weights/best.pt, 40.7MB

Validating /content/drive/MyDrive/doclayout-yolo-indic/output/finetune_indicdlp/finetune/weights/best.pt...
Ultralytics YOLOv0.0.2 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 94/94 [00:46<00:00,  2.01it/s]


3000
                   all       3000      46584      0.538      0.472      0.474      0.328
         advertisement       3000        408      0.563      0.797      0.662      0.573
                answer       3000        129      0.351     0.0388     0.0657     0.0511
                author       3000        602      0.547      0.399      0.424       0.28
         chapter-title       3000        186      0.546      0.414      0.474      0.306
          contact-info       3000        430      0.464      0.226      0.264      0.179
              dateline       3000        977       0.58      0.561      0.552      0.302
                figure       3000       2376      0.718      0.803      0.806      0.656
        figure-caption       3000        678      0.476       0.54      0.523      0.324
  first-level-question       3000       1172       0.56      0.608      0.589      0.469
                  flag       3000        116      0.546      0.491      0.476      0.313
                

## Next
- `src/finetuning/data_prep.py` — BaDLAD COCO->YOLO 9-class remap + IndicDLP yaml
- `src/finetuning/train_finetuning.py` — re-head to IndicDLP's 42 classes + fine-tune (final model)
- `src/finetuning/ablation.py` — the 3-5 ablation runs

(`src/utils/ontology.py` already exists: pretrain 9-class head, then **replace the head**
with IndicDLP's 42 classes at fine-tuning. Run its `inspect_indicdlp_categories()` first.)